# 🧪 BlogMS — Comments Tests (`test_comments.py`)

| # | Test | الوصف |
|---|------|-------|
| 1 | `test_create_comment_as_reader` | Reader يضيف تعليق |
| 2 | `test_create_nested_comment` | إضافة reply على تعليق |
| 3 | `test_list_comments_nested` | جلب التعليقات مع الـ replies |
| 4 | `test_comment_on_nonexistent_post` | تعليق على بوست مش موجود |
| 5 | `test_update_own_comment` | تعديل التعليق من صاحبه |
| 6 | `test_delete_comment_by_admin` | Admin يحذف أي تعليق |
| 7 | `test_delete_comment_by_other_user_forbidden` | منع حذف تعليق شخص تاني |

## 🛠️ Helper: `_create_post`

In [ ]:
def _create_post(client, token):
    """Helper بينشئ بوست ويرجع الـ ID — بيُستخدم في كل الـ tests."""
    return client.post(
        "/posts/",
        json={"title": "Test Post", "content": "Content"},
        headers={"Authorization": f"Bearer {token}"},
    ).json()["id"]

## 💬 `test_create_comment_as_reader`

In [ ]:
def test_create_comment_as_reader(client, author_token, reader_token):
    """✅ 201 · content صح · parent_id=None"""
    post_id = _create_post(client, author_token)
    resp = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Nice post!"},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    assert resp.status_code == 201
    assert resp.json()["content"] == "Nice post!"
    assert resp.json()["parent_id"] is None

## 💬 `test_create_nested_comment`

In [ ]:
def test_create_nested_comment(client, author_token, reader_token):
    """✅ 201 · parent_id صح"""
    post_id = _create_post(client, author_token)
    parent = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Top-level"},
        headers={"Authorization": f"Bearer {reader_token}"},
    ).json()
    resp = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Reply here!", "parent_id": parent["id"]},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    assert resp.status_code == 201
    assert resp.json()["parent_id"] == parent["id"]

## 💬 `test_list_comments_nested`

In [ ]:
def test_list_comments_nested(client, author_token, reader_token):
    """✅ total=1 top-level · replies متداخلة صح"""
    post_id = _create_post(client, author_token)
    parent_id = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Parent"},
        headers={"Authorization": f"Bearer {reader_token}"},
    ).json()["id"]
    client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Child", "parent_id": parent_id},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    resp = client.get(f"/posts/{post_id}/comments/")
    data = resp.json()
    assert data["total"] == 1
    assert len(data["items"][0]["replies"]) == 1
    assert data["items"][0]["replies"][0]["content"] == "Child"

## 💬 `test_comment_on_nonexistent_post`

In [ ]:
def test_comment_on_nonexistent_post(client, reader_token):
    """❌ 404 · بوست مش موجود"""
    resp = client.post(
        "/posts/9999/comments/",
        json={"content": "Ghost"},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    assert resp.status_code == 404

## ✏️ `test_update_own_comment`

In [ ]:
def test_update_own_comment(client, author_token, reader_token):
    """✅ 200 · content اتحدّث"""
    post_id = _create_post(client, author_token)
    comment_id = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Old"},
        headers={"Authorization": f"Bearer {reader_token}"},
    ).json()["id"]
    resp = client.put(
        f"/posts/{post_id}/comments/{comment_id}",
        json={"content": "Updated"},
        headers={"Authorization": f"Bearer {reader_token}"},
    )
    assert resp.status_code == 200
    assert resp.json()["content"] == "Updated"

## 🗑️ `test_delete_comment_by_admin`

In [ ]:
def test_delete_comment_by_admin(client, author_token, reader_token, admin_token):
    """✅ 204 · Admin يحذف أي تعليق"""
    post_id = _create_post(client, author_token)
    comment_id = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Delete me"},
        headers={"Authorization": f"Bearer {reader_token}"},
    ).json()["id"]
    resp = client.delete(
        f"/posts/{post_id}/comments/{comment_id}",
        headers={"Authorization": f"Bearer {admin_token}"},
    )
    assert resp.status_code == 204

## 🗑️ `test_delete_comment_by_other_user_forbidden`

In [ ]:
def test_delete_comment_by_other_user_forbidden(client, author_token, reader_token):
    """❌ 403 · يوزر تاني مش صاحب التعليق"""
    post_id = _create_post(client, author_token)
    comment_id = client.post(
        f"/posts/{post_id}/comments/",
        json={"content": "Mine"},
        headers={"Authorization": f"Bearer {reader_token}"},
    ).json()["id"]
    resp = client.delete(
        f"/posts/{post_id}/comments/{comment_id}",
        headers={"Authorization": f"Bearer {author_token}"},
    )
    assert resp.status_code == 403

## ▶️ تشغيل الـ Tests

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_comments.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)